<a href="https://colab.research.google.com/github/Mohammed-Taher6705/jigsaw-puzzle-matching/blob/main/Matching_with_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!git clone https://github.com/Mohammed-Taher6705/jigsaw-puzzle-matching.git

Cloning into 'jigsaw-puzzle-matching'...
remote: Enumerating objects: 1365, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 1365 (delta 21), reused 19 (delta 15), pack-reused 1334 (from 4)
Receiving objects: 100% (1365/1365), 449.78 MiB | 5.31 MiB/s, done.
Resolving deltas: 100% (103/103), done.


In [9]:
import zipfile
import os
import cv2
import numpy as np
from itertools import permutations
from skimage.metrics import structural_similarity as ssim
import shutil

In [10]:
zip_path = "/content/jigsaw-puzzle-matching/cropped_dataset.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
print("Folders inside extracted dataset:", os.listdir(extract_path))


Extracted to: /content
Folders inside extracted dataset: ['.config', 'cropped_dataset', 'jigsaw-puzzle-matching', 'sample_data']


In [7]:
!rm -rf /content/jigsaw-puzzle-matching

In [11]:
zip_path = "/content/jigsaw-puzzle-matching/complete_output.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
print("Folders inside extracted dataset:", os.listdir(extract_path))


Extracted to: /content
Folders inside extracted dataset: ['.config', 'complete_output', 'cropped_dataset', 'jigsaw-puzzle-matching', 'sample_data']


In [12]:
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import GridBox, Layout
import cv2
import os
import re

# -----------------------------
# Read image as RGB (for logic)
# -----------------------------
def read_rgb(path):
    img = cv2.imread(path)
    if img is None:
        return None
    return img[..., ::-1]  # BGR → RGB


# -----------------------------
# Encode correctly for display
# -----------------------------
def encode_jpg(rgb_img):
    bgr = rgb_img[..., ::-1]  # RGB → BGR for OpenCV
    return cv2.imencode(".jpg", bgr)[1].tobytes()


display(widgets.HTML("""
<style>
.output_wrapper, .widget-box, .jp-OutputArea {
    overflow-x: hidden !important;
}
</style>
"""))

header = widgets.HTML("""
<div style="text-align:center">
    <h2 style="margin-bottom:6px">🧩 Jigsaw Puzzle Viewer</h2>
    <p style="color:#666; font-size:13px">
        Interactive visualization of reconstructed puzzles
    </p>
    <hr>
</div>
""")

puzzle_type = widgets.Dropdown(
    options=['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8'],
    description='Puzzle',
    layout=widgets.Layout(width='100%')
)

puzzle_id = widgets.IntText(
    value=0,
    description='Puzzle ID',
    layout=widgets.Layout(width='100%')
)

load_btn = widgets.Button(
    description='Load Result',
    icon='image',
    button_style='success',
    layout=widgets.Layout(width='100%', height='40px')
)

status = widgets.HTML("<b>Status:</b> 🟢 Ready")

solver_panel = widgets.VBox(
    [header, puzzle_type, puzzle_id, load_btn, status],
    layout=widgets.Layout(
        width='100%',
        max_width='480px',
        padding='25px',
        border='1px solid #ddd',
        border_radius='10px',
        box_shadow='0 4px 12px rgba(0,0,0,0.08)'
    )
)

pieces_box = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='10px',
        width='100%',
        max_width='420px'
    )
)

image_box = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='12px',
        width='100%',
        max_width='420px'
    )
)

result_panel = widgets.VBox(
    [
        widgets.HTML("<h3 style='text-align:center'>Result Visualization</h3>"),
        widgets.HBox(
            [
                widgets.VBox([widgets.HTML("<b>Cropped Pieces</b>"), pieces_box],
                             layout=widgets.Layout(width='50%')),
                widgets.VBox([widgets.HTML("<b>Assembled Result</b>"), image_box],
                             layout=widgets.Layout(width='50%'))
            ],
            layout=widgets.Layout(gap='20px')
        )
    ],
    layout=widgets.Layout(
        display='none',
        width='100%',
        max_width='900px',
        padding='25px',
        border='1px solid #ddd',
        border_radius='10px',
        box_shadow='0 4px 12px rgba(0,0,0,0.08)'
    )
)

def on_load_clicked(b):
    image_box.clear_output()
    pieces_box.clear_output()
    result_panel.layout.display = 'none'

    ptype = puzzle_type.value
    pid = puzzle_id.value

    if pid < 0 or pid > 109:
        status.value = "<b>Status:</b> ❌ No image with this ID (0–109)"
        return

    if ptype == "puzzle_2x2":
        grid_size = 2
        tile_size = 80
    elif ptype == "puzzle_4x4":
        grid_size = 4
        tile_size = 65
    else:
        grid_size = 8
        tile_size = 55

    pieces_dir = f"/content/cropped_dataset/{ptype}"
    tiles = []

    for fname in os.listdir(pieces_dir):
        m = re.search(r"^(\d+)_r(\d+)_c(\d+)", fname)
        if m and int(m.group(1)) == pid:
            tiles.append((int(m.group(2)), int(m.group(3)), fname))

    if not tiles:
        status.value = "<b>Status:</b> ❌ No cropped pieces found"
        return

    tiles.sort(key=lambda x: (x[0], x[1]))

    result_path = f"/content/complete_output/{ptype}/{pid}.jpg"
    if os.path.exists(result_path):
        img = read_rgb(result_path)
        if img is not None:
            result_panel.layout.display = 'block'
            with image_box:
                display(widgets.Image(
                    value=encode_jpg(img),
                    format="jpg",
                    width=360
                ))

    all_imgs = []
    for r, c, fname in tiles:
        tile = read_rgb(os.path.join(pieces_dir, fname))
        if tile is None:
            continue

        all_imgs.append(
            widgets.Image(
                value=encode_jpg(tile),
                format="jpg",
                width=tile_size
            )
        )

    with pieces_box:
        display(
            GridBox(
                children=all_imgs,
                layout=Layout(
                    grid_template_columns=f"repeat({grid_size}, {tile_size}px)",
                    grid_gap="4px"
                )
            )
        )

    status.value = "<b>Status:</b> ✅ Loaded successfully"

load_btn.on_click(on_load_clicked)

display(
    widgets.VBox(
        [solver_panel, result_panel],
        layout=widgets.Layout(
            width='100%',
            align_items='center',
            gap='30px',
            margin='30px auto'
        )
    )
)

HTML(value='\n<style>\n.output_wrapper, .widget-box, .jp-OutputArea {\n    overflow-x: hidden !important;\n}\n…